# Black-box top-k sources and robust targets

Generate target→top sources and source→top robust targets using the shared transfer-view configuration.

# Black-box transfer analysis configuration

All three notebooks use the same `TransferViewConfig` object, so the attack matrix, source--target scatter, and top-k tables are directly comparable.

Key knobs:

- `source_grain` / `target_grain`: `family`, `architecture`, or `variant`.
- `comparison_mode`:
  - `inter_variant`: exclude exact same variant only.
  - `inter_architecture`: exclude same architecture.
  - `inter_model`: exclude same family.
- `variant_aggregate_mode`: how variants are collapsed into architectures/families: `mean`, `median`, `best_damage`, `best_robust`.
- `architecture_aggregate_mode`: how architectures are collapsed into families: `mean`, `median`, `best_damage`, `best_robust`.
- `split_ansatz_as_architecture`: when `True`, VQM/HVQL/etc. ansatz variants become architecture labels such as `VQM-Strong` or `HVQL-Random`.
- `blacklist_contains` and `model_blacklist` apply to all outputs.


In [18]:

from pathlib import Path
import sys

try:
    from analytics.paths import find_project_root, output_dir
except ModuleNotFoundError:
    # Allows running the notebook from a copied location.
    repo_root = Path.cwd()
    while repo_root != repo_root.parent and not (repo_root / "src").exists():
        repo_root = repo_root.parent
    sys.path.insert(0, str(repo_root / "src"))
    from analytics.paths import find_project_root, output_dir

from analytics.attacks.transfer_role_tools import (
    TransferViewConfig,
    VIEW_TEMPLATES,
    describe_config,
    get_template,
    list_templates,
    load_or_build_pair_metrics,
    build_attack_matrix,
    build_source_target_scatter,
    build_top_sources_by_target,
    build_top_targets_by_source,
    plot_attack_matrix,
    plot_source_target_scatter,
    save_figure,
    write_outputs,
)

ROOT = find_project_root()
PAIR_METRICS_CACHE = output_dir(ROOT, "attack_transfer_role_tables") / "pair_metrics.csv"
OUT_DIR = output_dir(ROOT, "blackbox_transfer_views")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Template shortcuts:
# list_templates()
TEMPLATE_NAME = "architecture_mean"

CONFIG = get_template(
    TEMPLATE_NAME,
    source_grain="architecture",
    target_grain="variant",
    comparison_mode="inter_architecture",
    metric="Transfer ASR",
    top_k=3,
    variant_aggregate_mode="mean",
    architecture_aggregate_mode="mean",
    split_ansatz_as_architecture=False,
    # Exact model names to exclude.
    model_blacklist=tuple(),
    # Substring rules, useful for quick exclusions such as ("HQuanv",).
    blacklist_contains=tuple(),
)

display(describe_config(CONFIG))


,value
source_grain,architecture
target_grain,variant
comparison_mode,inter_architecture
metric,Transfer ASR
top_k,3
variant_aggregate_mode,mean
architecture_aggregate_mode,mean
split_ansatz_as_architecture,False
include_no_entangling_as_ansatz,True
model_blacklist,()


In [6]:
display(list_templates())

,template,source_grain,target_grain,comparison_mode,metric,top_k,variant_aggregate_mode,architecture_aggregate_mode,split_ansatz_as_architecture,include_no_entangling_as_ansatz,model_blacklist,blacklist_contains,source_family_filter,target_family_filter,source_architecture_filter,target_architecture_filter,source_contains,target_contains
0,variant_full,variant,variant,inter_variant,Transfer ASR,3,mean,mean,False,True,(),(),None,None,None,None,None,None
1,architecture_mean,architecture,architecture,inter_architecture,Transfer ASR,3,mean,mean,False,True,(),(),None,None,None,None,None,None
2,architecture_best_damage,architecture,architecture,inter_architecture,Transfer ASR,3,best_damage,mean,False,True,(),(),None,None,None,None,None,None
3,architecture_best_robust,architecture,architecture,inter_architecture,Transfer ASR,3,best_robust,mean,False,True,(),(),None,None,None,None,None,None
4,architecture_ansatz_mean,architecture,architecture,inter_architecture,Transfer ASR,3,mean,mean,True,True,(),(),None,None,None,None,None,None
5,family_mean_architecture_mean_variant,family,family,inter_variant,Transfer ASR,3,mean,mean,False,True,(),(),None,None,None,None,None,None
6,family_mean_architecture_best_damage_variant,family,family,inter_variant,Transfer ASR,3,best_damage,mean,False,True,(),(),None,None,None,None,None,None
7,family_mean_architecture_best_robust_variant,family,family,inter_variant,Transfer ASR,3,best_robust,mean,False,True,(),(),None,None,None,None,None,None
8,inter_family_architecture_mean,architecture,architecture,inter_model,Transfer ASR,3,mean,mean,False,True,(),(),None,None,None,None,None,None
9,top_variants_vs_target_architecture,variant,architecture,inter_architecture,Transfer ASR,3,mean,mean,False,True,(),(),None,None,None,None,None,None


In [15]:

pair_df = load_or_build_pair_metrics(
    pair_metrics_cache=PAIR_METRICS_CACHE,
    rebuild=False,
)
print(f"Loaded pair metrics: {pair_df.shape[0]:,} rows × {pair_df.shape[1]:,} columns")


Loaded pair metrics: 1,089 rows × 24 columns


In [16]:

# Most damaging sources for each target.
top_sources_wide, top_sources_long = build_top_sources_by_target(
    pair_df,
    CONFIG,
    rank_mode="damage",
)
print("Top damaging sources by target")
display(top_sources_wide)


Top damaging sources by target


,target,top 1 source (Transfer ASR),top 2 source (Transfer ASR),top 3 source (Transfer ASR)
0,Classical/CNN,"Classical/MLP/hidden_sizes=[128,64] (82.7%)","Classical/MLP/hidden_sizes=[64,32] (82.4%)","Classical/MLP/hidden_sizes=[32,16] (82.0%)"
1,Classical/MLP,Classical/CNN/LeNet (88.9%),Hybrid/HQuanv (86.4%),"Classical/CNN/conv_channels=[32,64], fc_sizes=..."
2,Classical/MLPMixer,Classical/CNN/LeNet (81.9%),"Classical/MLP/hidden_sizes=[128,64] (81.2%)","Classical/CNN/conv_channels=[32,64], fc_sizes=..."
3,Classical/ResNet18,Hybrid/HVQL/residuals (77.2%),Classical/CNN/LeNet (76.8%),"Hybrid/HPVQL/qblocks=10, qdepth=50, ansatz=str..."
4,Hybrid/HPVQL,"Classical/MLP/hidden_sizes=[1024,512,64,64] (7...","Classical/CNN/conv_channels=[32,64], fc_sizes=...",Hybrid/HVQL/residuals (75.2%)
5,Hybrid/HQE,Classical/CNN/LeNet (89.1%),"Classical/MLP/hidden_sizes=[128,64] (87.8%)","Classical/MLP/hidden_sizes=[64,32] (87.0%)"
6,Hybrid/HQuanv,Classical/CNN/LeNet (86.5%),"Classical/MLP/hidden_sizes=[128,64] (86.3%)","Classical/MLP/hidden_sizes=[64,32] (85.8%)"
7,Hybrid/HVQL,"Classical/CNN/conv_channels=[32,64], fc_sizes=...","Classical/MLP/hidden_sizes=[1024,512,64,64] (7...",Classical/CNN/LeNet (73.3%)
8,Hybrid/HVQM,"Classical/MLP/hidden_sizes=[1024,512,64,64] (7...",Classical/CNN/LeNet (75.4%),"Classical/CNN/conv_channels=[32,64], fc_sizes=..."
9,Quantum/VQM,Classical/CNN/LeNet (70.9%),"Classical/MLP/hidden_sizes=[64,32] (68.8%)","Classical/MLP/hidden_sizes=[4,4] (68.7%)"


In [19]:

# Most robust / least damaged targets for each source.
top_targets_wide, top_targets_long = build_top_targets_by_source(
    pair_df,
    CONFIG,
    rank_mode="robust",
)
print("Top robust targets by source")
display(top_targets_wide)


Top robust targets by source


,source,top 1 robust target (Transfer ASR),top 2 robust target (Transfer ASR),top 3 robust target (Transfer ASR)
0,Classical/CNN,"Quantum/VQM/ansatz=strong, qdepth=200 (63.1%)","Quantum/VQM/ansatz=strong, qdepth=100 (63.4%)","Quantum/VQM/ansatz=strong, qdepth=50 (64.5%)"
1,Classical/MLP,"Quantum/VQM/ansatz=basic, qdepth=50 (64.2%)","Quantum/VQM/ansatz=strong, qdepth=100 (66.3%)","Quantum/VQM/ansatz=strong, qdepth=200 (66.4%)"
2,Classical/MLPMixer,"Quantum/VQM/ansatz=strong, qdepth=200 (49.6%)","Quantum/VQM/ansatz=strong, qdepth=50 (50.0%)","Quantum/VQM/ansatz=strong, qdepth=100 (50.5%)"
3,Classical/ResNet18,"Quantum/VQM/ansatz=strong, qdepth=100 (48.4%)","Quantum/VQM/ansatz=strong, qdepth=200 (49.7%)","Quantum/VQM/ansatz=strong, qdepth=50 (52.2%)"
4,Hybrid/HPVQL,"Quantum/VQM/ansatz=strong, qdepth=200 (47.8%)","Quantum/VQM/ansatz=strong, qdepth=100 (48.0%)","Quantum/VQM/ansatz=strong, qdepth=50 (49.5%)"
5,Hybrid/HQE,"Quantum/VQM/ansatz=strong, qdepth=200 (36.7%)","Quantum/VQM/ansatz=strong, qdepth=50 (37.3%)","Quantum/VQM/ansatz=strong, qdepth=100 (37.4%)"
6,Hybrid/HQuanv,"Quantum/VQM/ansatz=strong, qdepth=200 (64.9%)","Quantum/VQM/ansatz=strong, qdepth=100 (65.0%)","Quantum/VQM/ansatz=basic, qdepth=50 (65.3%)"
7,Hybrid/HVQL,"Quantum/VQM/ansatz=strong, qdepth=100 (46.7%)","Quantum/VQM/ansatz=strong, qdepth=200 (46.8%)","Quantum/VQM/ansatz=strong, qdepth=50 (48.1%)"
8,Hybrid/HVQM,"Classical/MLP/hidden_sizes=[1024,512,64,64] (5...","Classical/CNN/conv_channels=[32,64], fc_sizes=...","Quantum/VQM/ansatz=strong, qdepth=50 (56.9%)"
9,Quantum/VQM,"Classical/MLP/hidden_sizes=[1024,512,64,64] (4...","Classical/CNN/conv_channels=[32,64], fc_sizes=...","Hybrid/HPVQL/qblocks=10, qdepth=50, ansatz=str..."


In [ ]:

prefix = f"topk_{CONFIG.source_grain}_to_{CONFIG.target_grain}_{CONFIG.comparison_mode}_{CONFIG.metric.replace(' ', '_')}"
outputs = write_outputs(
    OUT_DIR,
    prefix=prefix,
    top_sources_wide=top_sources_wide,
    top_sources_long=top_sources_long,
    top_targets_wide=top_targets_wide,
    top_targets_long=top_targets_long,
)
outputs
